In [ ]:
# YZ50 Demo Day - 6: modeli konuştur, ses örneğiyle (voice prompt)
# Cümlenin önüne kendi gerçek kaydımı (metin + ses token'ları) koyar; model sesi oradan devam ettirir.
# Yeni eğitim yok. Çıktı: Drive yz50-demo-day/samples/speak/*.wav

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, math, wave
import numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from transformers import MimiModel

In [ ]:
# Sözlük: özel token'lar + metin karakterleri + ses token'ları. Bütün notebook'larda aynı.
import unicodedata
PAD, BOS, SEP, EOS, MALE, FEMALE, ME = 0, 1, 2, 3, 4, 5, 6
CHARS = " abcçdefgğhıijklmnoöpqrsştuüvwxyzâîû0123456789.,?!'-:"
CHAR_BASE = 16
stoi = {c: CHAR_BASE + i for i, c in enumerate(CHARS)}
SR, FRAME_RATE = 24000, 12.5             # Mimi codec: 24 kHz ses, saniyede 12.5 kare
N_CB, CB_SIZE = 8, 2048                  # her karede 8 codebook, her biri 2048 kod
AUDIO_BASE = 128
VOCAB = AUDIO_BASE + N_CB * CB_SIZE      # 16512
TOK_PER_S = int(N_CB * FRAME_RATE)       # saniyede 100 ses token'ı
assert CHAR_BASE + len(CHARS) <= AUDIO_BASE

_MAP = str.maketrans({"’": "'", "‘": "'", "“": "", "”": "", '"': "", "–": "-", "—": "-", "…": "."})
def norm_text(s):
    s = unicodedata.normalize("NFC", s).replace("I", "ı").replace("İ", "i").lower().translate(_MAP)
    return " ".join("".join(c for c in s if c in stoi).split())
def encode_text(s):
    return [stoi[c] for c in norm_text(s)]
# Bir örnek: [BOS, ses etiketi, metin karakterleri..., SEP, kare0 cb0..cb7, kare1 cb0..cb7, ..., EOS]

In [ ]:
ROOT = '/content/drive/MyDrive/yz50-demo-day'
CKPTS = {'best': f'{ROOT}/ckpt/finetune_read_best.pt', 'last': f'{ROOT}/ckpt/finetune_read_last.pt'}
PROMPT_FILE = f'{ROOT}/tokens/me_read/train_read.npz'
N_PROMPTS = 3                          # kaç farklı kayıt denenecek
TEXTS = ['Merhaba, ben Alperen ve bu model benim sesimle konuşuyor.',
         'Bu sesi kaydetmedim, bu sesi modelim üretti.']
SEEDS, MAX_SECONDS, TEMP, TOP_K = [0, 1], 10, 0.8, 80
DEVICE, AMP = 'cuda', True
BLOCK, N_LAYER, N_HEAD, N_EMBD, DROPOUT = 2048, 12, 12, 768, 0.0     # yalnız model.py için; asıl ayar checkpoint'ten

In [ ]:
# Model: GPT-2 mimarisi, sıfırdan
class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_attn = nn.Linear(c['n_embd'], 3 * c['n_embd'])
        self.c_proj = nn.Linear(c['n_embd'], c['n_embd']); self.c_proj.RESID = True
        self.n_head, self.dropout = c['n_head'], c['dropout']
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_fc = nn.Linear(c['n_embd'], 4 * c['n_embd'])
        self.c_proj = nn.Linear(4 * c['n_embd'], c['n_embd']); self.c_proj.RESID = True
        self.drop = nn.Dropout(c['dropout'])
    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x), approximate='tanh')))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1, self.attn = nn.LayerNorm(c['n_embd']), CausalSelfAttention(c)
        self.ln_2, self.mlp = nn.LayerNorm(c['n_embd']), MLP(c)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.wte = nn.Embedding(c['vocab'], c['n_embd'])
        self.wpe = nn.Embedding(c['block'], c['n_embd'])
        self.h = nn.ModuleList([Block(c) for _ in range(c['n_layer'])])
        self.ln_f = nn.LayerNorm(c['n_embd'])
        self.lm_head = nn.Linear(c['n_embd'], c['vocab'], bias=False)
        self.wte.weight = self.lm_head.weight                      # weight tying
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02 * (2 * self.c['n_layer']) ** -0.5 if hasattr(m, 'RESID') else 0.02
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None, last_only=False):
        B, T = idx.shape
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        for b in self.h: x = b(x)
        x = self.ln_f(x)
        if last_only: return self.lm_head(x[:, -1, :]), None
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        return logits, loss

cfg = dict(vocab=VOCAB, block=BLOCK, n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT)

In [ ]:
OUT = f'{ROOT}/samples/speak'; os.makedirs(OUT, exist_ok=True)
mimi = MimiModel.from_pretrained('kyutai/mimi').to(DEVICE).eval()
def autocast(): return torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=AMP and DEVICE == 'cuda')

# Ses örnekleri: okuma kayıtlarımdan 3–6 sn'lik klipler
d = np.load(PROMPT_FILE); tok, lens = d['tokens'], d['lens']; starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
def text_of(q): return ''.join(CHARS[c - CHAR_BASE] for c in q[2:int(np.argmax(q == SEP))])
cand = [k for k in range(len(lens)) if 300 + 60 <= lens[k] <= 600 + 80]
pick = [cand[j] for j in np.linspace(0, len(cand) - 1, N_PROMPTS).astype(int)]
prompts = [tok[starts[k]:starts[k] + lens[k]].astype(np.int64) for k in pick]
for k, p in enumerate(prompts): print(f'örnek {k}: {len(p)} token | "{text_of(p)}"')

@torch.no_grad()
def speak(model, text, prefix=None, seed=0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    head = ([] if prefix is None else list(prefix)) + [BOS, ME] + encode_text(text) + [SEP]
    x = torch.tensor([head], device=DEVICE); out = []
    while len(out) < int(MAX_SECONDS * TOK_PER_S) and x.shape[1] < model.c['block']:
        with autocast(): logits, _ = model(x, last_only=True)
        logits = logits[0].float(); cb = len(out) % N_CB; lo = AUDIO_BASE + cb * CB_SIZE
        mask = torch.full_like(logits, float('-inf')); mask[lo:lo + CB_SIZE] = 0
        if cb == 0 and len(out) >= N_CB: mask[EOS] = 0
        v, ix = torch.topk((logits + mask) / TEMP, TOP_K)
        t = ix[torch.multinomial(F.softmax(v, dim=-1), 1, generator=g)]
        if t.item() == EOS: break
        out.append(t.item() - lo); x = torch.cat([x, t.view(1, 1)], 1)
    n = len(out) // N_CB
    if n < 2: return np.zeros(SR // 10, np.float32)
    codes = torch.tensor(out[:n * N_CB], device=DEVICE).view(n, N_CB).T[None]
    return mimi.decode(codes).audio_values[0, 0].float().cpu().numpy()

def save_wav(path, a):
    with wave.open(path, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes((np.clip(a, -1, 1) * 32767).astype(np.int16).tobytes())

In [ ]:
# Konuştur: her checkpoint × (örneksiz + her ses örneği) × her cümle × her seed
from IPython.display import Audio, display
for name, path in CKPTS.items():
    if not os.path.exists(path): print('yok:', path); continue
    ck = torch.load(path, map_location=DEVICE)
    model = GPT(ck['cfg']).to(DEVICE).eval(); model.load_state_dict(ck['model'])
    for ti, text in enumerate(TEXTS):
        for pi, pre in [(-1, None)] + list(enumerate(prompts)):
            for s in SEEDS:
                a = speak(model, text, pre, s)
                f = f'{name}_t{ti}_{"noprompt" if pi < 0 else f"prompt{pi}"}_s{s}.wav'
                save_wav(f'{OUT}/{f}', a)
                print(f'{f}  ({len(a) / SR:.1f} sn)  {text}'); display(Audio(a, rate=SR))
    del model; torch.cuda.empty_cache() if DEVICE == 'cuda' else None
print('hepsi:', OUT)